# 00. Dataset preparation - GSE69914

### Libraries

In [ ]:
!pip install GEOparse


In [3]:
import pandas as pd
import GEOparse
import re
import csv


## Loading and Initial Cleaning of GEO Series Matrix Data

In [4]:
# Read txt file
input_path  = "/kaggle/input/gse69914-series-matrix-txt/GSE69914_series_matrix.txt" ## CHANGE HERE!! ## 
output_path = "GSE69914_series_matrix_vr2.csv" ## CHANGE HERE!! ##

# Read the data into a pandas DataFrame
df = pd.read_csv(
    input_path,
    sep="\t",           # Data fields are separated by tabs in GEO series matrix files
    compression="infer",
    skiprows=73,        # Skip the GEO file header (metadata lines)
    comment="!",        # Ignore the closing line "!series_matrix_table_end" and any other metadata lines
    engine="c"          # Use the C engine for faster processing
)

# Ensure the first column is indeed named ID_REF (it is sometimes already named this way)
if df.columns[0] != "ID_REF":
    # Rename the first column to "ID_REF", which typically contains probe or gene identifiers
    df.rename(columns={df.columns[0]: "ID_REF"}, inplace=True)

# Numeric cast only on sample columns (leave ID_REF as string)
# Iterate over all columns starting from the second one (index 1), as the first one is 'ID_REF'
for c in df.columns[1:]:
    # Convert the column data type to numeric. 'errors="coerce"' will turn non-numeric values 
    # (like missing data markers, if any are left) into NaN.
    df[c] = pd.to_numeric(df[c], errors="coerce")
    

Letto
Prima colonna
Colonne
✅ Salvato: GSE69914_series_matrix_vr2.csv


#### Data Manipulation and Transposition

In [5]:
output_path = "/kaggle/working/GSE69914_series_matrix_vr2.csv" ## CHANGE HERE!! ##

# Load the previously generated CSV (the first column is the index: CpG/ID_REF)
# I load the processed file where the first column (ID_REF) should be set as the DataFrame index.
df = pd.read_csv(output_path, index_col=0)

# Force numeric conversion on all columns (beta values), non-numeric values -> NaN
# Although this was partially done in the previous step, this ensures all values are numeric, 
# converting any unexpected remaining non-numeric values (if any) to 'Not a Number' (NaN).
df = df.apply(pd.to_numeric, errors="coerce")

# Transpose the DataFrame: rows = Sample, columns = CpG
# Transposing is a common step in gene expression/methylation analysis:
# - Original (df): Rows are probes/features (CpG/ID_REF), Columns are samples.
# - Transposed (df_T): Rows are samples, Columns are probes/features (CpG/ID_REF).
# This format (samples as rows) is generally required for machine learning and statistical modeling tools.
df_T = df.T

# Display the first few rows of the transposed DataFrame
df_T.head()


,GSM1712367,GSM1712368,GSM1712369,GSM1712370,GSM1712371,GSM1712372,GSM1712373,GSM1712374,GSM1712375,GSM1712376,...,GSM1712770,GSM1712771,GSM1712772,GSM1712773,GSM1712774,GSM1712775,GSM1712776,GSM1712777,GSM1712778,GSM1712779
ID_REF,,,,,,,,,,,,,,,,,,,,,
cg00000029,0.258254,0.197553,0.275187,0.150849,0.240538,0.249905,0.378757,0.474699,0.363260,0.332967,...,0.273695,0.204416,0.250100,0.117792,0.079539,0.242573,0.117149,0.201398,0.216148,0.569178
cg00000108,0.986116,0.981426,0.972137,0.984434,0.987393,0.958457,0.984781,0.988312,0.971297,0.981409,...,0.978810,0.955072,0.972645,0.971132,0.982360,0.973398,0.926971,0.974608,0.976362,0.984590
cg00000109,0.889916,0.826830,0.839431,0.950852,0.897285,0.814509,0.898999,0.903172,0.920438,0.856352,...,0.844270,0.878550,0.852844,0.895050,0.930519,0.900096,0.625678,0.810017,0.889987,0.919995
cg00000165,0.247964,0.343906,0.216030,0.576088,0.616293,0.238897,0.243838,0.460061,0.429162,0.195441,...,0.224468,0.303041,0.237599,0.273763,0.903569,0.367987,0.380380,0.246235,0.410192,0.282906
cg00000236,0.902621,0.868874,0.837979,0.931236,0.915173,0.827905,0.905214,0.918090,0.886167,0.956775,...,0.929749,0.932601,0.860249,0.937596,0.939224,0.930300,0.784228,0.908534,0.911480,0.923068


### Data Inspection and Quality Check

In [14]:
# Dataset dimensions
# Print the number of rows (samples) and columns (features/CpG sites) in the transposed DataFrame.
# This gives an immediate overview of the dataset size for analysis.
print(f"Dataset shape: {df_T.shape}")

# Check for missing values
# Calculate the total number of missing values (NaNs) across the *entire* DataFrame.
# I sum the missing values per column (df_T.isnull().sum()) and then sum the resulting series.
print(f"Missing values per columns: {sum(df_T.isnull().sum())}")


Dataset shape: (407, 485512)
Missing values per columns: 0


## Fetching and Assigning Sample Labels (Metadata Extraction)

In [18]:
# To add the labels
# Download GSE69914 metadata from GEO
# I use GEOparse to download the entire GEO entry for GSE69914, including all sample (GSM) metadata.
gse = GEOparse.get_GEO("GSE69914", destdir=".")

# Extract metadata text for each GSM (Sample)
meta = []
# I iterate through all individual sample entries (GSMs) within the downloaded GEO object.
for gsm_name, gsm in gse.gsms.items():
    # Merge all metadata key-value pairs into a single, searchable string, converted to lowercase.
    text = " | ".join([f"{k}: {v}" for k, v in gsm.metadata.items()]).lower()
    
    # Search for the specific string containing "status(...): X" where X is the numeric code (0-4).
    # This regex is tailored to extract the relevant biological status code from the metadata text.
    match = re.search(r"status.*?:\s*([0-4])", text)
    # Extract the matched code and convert it to an integer; otherwise, assign None if no match is found.
    status_code = int(match.group(1)) if match else None
    
    # Append the results (GSM name, status code, and full metadata text) to the list.
    meta.append({
        "gsm": gsm_name,
        "status_code": status_code,
        "meta_text": text
    })

# Convert the list of metadata dictionaries into a pandas DataFrame for easier manipulation.
meta_df = pd.DataFrame(meta)

# Map numbers → human-readable labels
# Define a mapping dictionary to convert the cryptic numeric status codes into meaningful biological labels.
status_map = {
    0: "Normal",
    1: "Normal-Adjacent",
    2: "Breast Cancer",
    3: "Normal-BRCA1",
    4: "Cancer-BRCA1"
}
# Create a new column 'status_label' by applying the map to the extracted 'status_code' column.
meta_df["status_label"] = meta_df["status_code"].map(status_map)

# Save to a clean CSV 
# I save the essential columns (sample ID, numeric code, and label) to a separate CSV file.
meta_df[["gsm", "status_code", "status_label"]].to_csv("GSE69914_status_labels.csv", index=False)

# Print a confirmation message and show the distribution of the newly created labels.
print("✅ Saved: GSE69914_status_labels.csv")
print(meta_df["status_label"].value_counts(dropna=False))


03-Nov-2025 13:40:26 DEBUG utils - Directory . already exists. Skipping.
03-Nov-2025 13:40:26 INFO GEOparse - Downloading ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE69nnn/GSE69914/soft/GSE69914_family.soft.gz to ./GSE69914_family.soft.gz
100%|██████████| 2.29G/2.29G [01:37<00:00, 25.2MB/s]   
03-Nov-2025 13:42:04 DEBUG downloader - Size validation passed
03-Nov-2025 13:42:04 DEBUG downloader - Moving /tmp/tmp797srjmi to /kaggle/working/GSE69914_family.soft.gz
03-Nov-2025 13:42:07 DEBUG downloader - Successfully downloaded ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE69nnn/GSE69914/soft/GSE69914_family.soft.gz
03-Nov-2025 13:42:07 INFO GEOparse - Parsing ./GSE69914_family.soft.gz: 
03-Nov-2025 13:42:07 DEBUG GEOparse - DATABASE: GeoMiame
03-Nov-2025 13:42:07 DEBUG GEOparse - SERIES: GSE69914
03-Nov-2025 13:42:07 DEBUG GEOparse - PLATFORM: GPL16304
/usr/local/lib/python3.11/dist-packages/GEOparse/GEOparse.py:401: DtypeWarning: Columns (16,17) have mixed types. Specify dtype option on import or

✅ Salvato: GSE69914_status_labels.csv
status_label
Breast Cancer      305
Normal              50
Normal-Adjacent     42
Normal-BRCA1         7
Cancer-BRCA1         3
Name: count, dtype: int64


## Merging Feature Matrix with Numeric Sample Labels

In [ ]:
# Define file paths
MATRIX_CSV = "/kaggle/working/GSE69914_series_matrix_T_vr2.csv"  # The Transposed Matrix (Row=Sample, Col=CpG) ## CHANGE HERE!! ##
LABELS_CSV = "/kaggle/working/GSE69914_meta_df.csv"              # Columns: gsm, status_code ## CHANGE HERE!! ##
OUT_CSV    = "/kaggle/working/GSE69914_beta_with_labels_numeric.csv" ## CHANGE HERE!! ##

# Load the labels into a dictionary {gsm -> code}
label_map = {}
# I open the labels CSV to read the sample IDs and their corresponding numeric status codes.
with open(LABELS_CSV, newline="", encoding="utf-8") as f:
    # Use DictReader to easily access columns by their name ('gsm', 'status_code').
    r = csv.DictReader(f)
    # If column names were different, I would adapt them here (e.g., using r.fieldnames).
    for row in r:
        gsm = row.get("gsm")
        code = row.get("status_code")
        # Ensure both GSM and code are present and not empty before adding to the map.
        if gsm is not None and code is not None and code != "":
            # Store the label as an integer.
            label_map[gsm] = int(code)

# Stream the matrix file: copy each row + append the label, writing directly to disk
# This process avoids loading the potentially huge data matrix into RAM all at once.
with open(MATRIX_CSV, newline="", encoding="utf-8") as fin, \
     open(OUT_CSV, "w", newline="", encoding="utf-8") as fout:

    # Set up CSV reader for the input matrix and writer for the output file.
    reader = csv.reader(fin)
    writer = csv.writer(fout)

    # Read and modify the header: I add "label" as the LAST column
    try:
        header = next(reader)
    except StopIteration:
        # Handle case where file is empty
        header = []
    
    # The first element of the header (header[0]) should be the sample ID (e.g., "id_campione" or "ID_REF").
    # Write the new header, including the 'label' column.
    writer.writerow(header + ["label"])

    # Process data rows
    for row in reader:
        # The sample ID is expected to be in the very first column.
        sample_id = row[0]                              # first column = sample ID (GSM...)
        # Look up the corresponding numeric code. Returns None if the sample ID is missing from the label map.
        code = label_map.get(sample_id)                 # None if missing
        # Write the original row data followed by the extracted label.
        writer.writerow(row + [code])

print(f"✅ Saved: {OUT_CSV}")


In [1]:


MATRIX_CSV = "/kaggle/working/GSE69914_series_matrix_T_vr2.csv"   # Sample x CpG (riga=campione, col=cpg)
LABELS_CSV = "/kaggle/working/GSE69914_meta_df.csv"               # colonne: gsm, status_code
OUT_CSV    = "/kaggle/working/GSE69914_beta_with_labels_numeric.csv"

# 1) Carica i label in un dict {gsm -> code}
label_map = {}
with open(LABELS_CSV, newline="", encoding="utf-8") as f:
    r = csv.DictReader(f)
    # Se i nomi colonne fossero diversi, adattali qui
    for row in r:
        gsm = row.get("gsm")
        code = row.get("status_code")
        if gsm is not None and code is not None and code != "":
            label_map[gsm] = int(code)

# 2) Stream del file matrice: copia riga + label in coda, scrivendo subito su disco
with open(MATRIX_CSV, newline="", encoding="utf-8") as fin, \
     open(OUT_CSV, "w", newline="", encoding="utf-8") as fout:

    reader = csv.reader(fin)
    writer = csv.writer(fout)

    # header: aggiungo "label" come ULTIMA colonna
    header = next(reader)
    # header[0] deve essere l'id campione (es. "id_campione" o "ID_REF" o simile)
    writer.writerow(header + ["label"])

    # righe dati
    for row in reader:
        sample_id = row[0]                   # prima colonna = id campione (GSM...)
        code = label_map.get(sample_id)      # None se manca
        writer.writerow(row + [code])

print(f"✅ Salvato: {OUT_CSV}")


✅ Salvato: /kaggle/working/GSE69914_beta_with_labels_numeric.csv


In [ ]:
import pandas as pd
import csv
# Ensure I have the 'pyarrow' library installed for Parquet functionality: !pip install pyarrow

# --- Define File Paths ---
MATRIX_CSV = "/kaggle/working/GSE69914_series_matrix_T_vr2.csv"  # Sample x CpG (Row=Sample, Col=CpG)
LABELS_CSV = "/kaggle/working/GSE69914_meta_df.csv"              # Columns: gsm, status_code

# The final output path for the highly efficient Parquet file
PARQUET_OUT_CSV = "/kaggle/working/GSE69914_beta_with_labels_numeric.parquet"

# --- 1) Load the labels into a dictionary {gsm -> code} ---
# I still use the efficient csv standard library to load the small label file into a map.
label_map = {}
with open(LABELS_CSV, newline="", encoding="utf-8") as f:
    r = csv.DictReader(f)
    for row in r:
        gsm = row.get("gsm")
        code = row.get("status_code")
        if gsm is not None and code is not None and code != "":
            # Store the label as an integer for memory efficiency.
            label_map[gsm] = int(code)

# --- 2) Load the large data matrix into a pandas DataFrame ---
# Loading is necessary here to enable the merge and the subsequent Parquet write.
X = pd.read_csv(MATRIX_CSV, index_col=0)  # Index is set to the sample ID (GSM...)

# --- 3) Add the numeric 'label' column ---
# I create a pandas Series from the label map to align it with the DataFrame index.
label_series = pd.Series(label_map)

# I insert the 'label' column at the start (position 0) by mapping the index (sample IDs).
# Using 'Int8' ensures the label column is memory-efficient and handles potential missing samples (NaN).
X.insert(0, "label", X.index.to_series().map(label_series).astype("Int8"))

# --- 4) Optimize Data Types and Write to Parquet ---
# Convert the large float columns (CpG values) from the default float64 to float32.
# This significantly reduces file size and memory usage for the final dataset.
data_cols = X.columns.drop('label', errors='ignore')
X[data_cols] = X[data_cols].astype("float32")

# Write the final optimized DataFrame to the column-oriented Parquet format.
# 'index=True' saves the sample ID index as a column in the Parquet file.
X.to_parquet(PARQUET_OUT_CSV, index=True)

print(f"✅ Merged data and saved directly to Parquet: {PARQUET_OUT_CSV}")

# (Optional) quick check on the final label counts
print("\nFinal label counts:")
print(X["label"].value_counts(dropna=False))

## Final Data Loading, Memory Optimization, and Saving

In [ ]:
# Define the path to the combined CSV file containing beta values and numeric labels.
output_path_labels = "/kaggle/working/GSE69914_beta_with_labels_numeric.csv" ## CHANGE HERE!! ##

# Define the new output path for the memory-optimized file.
optimized_output_path = "/kaggle/working/GSE69914_beta_with_labels_float32.csv" ## CHANGE HERE!! ##

# Load the previously generated CSV (the first column is the sample ID index)
# I load the processed file where the first column (the sample ID 'id_campione') is set as the DataFrame index.
df_with_labels = pd.read_csv(output_path_labels, index_col=0)

# I select all columns EXCEPT the 'label' column, which should remain an integer type.
data_cols = df_with_labels.columns.drop('label', errors='ignore')

# I apply the memory-saving conversion (float64 -> float32) only to the data columns (CpG sites).
df_with_labels[data_cols] = df_with_labels[data_cols].astype("float32")

print(f"Original data types after load: {df_with_labels.dtypes.head(2)}")
print(f"Data types after conversion: {df_with_labels[data_cols].dtypes.head(2)}")

# Save the memory-optimized dataset
# Save the DataFrame to a new CSV file. The index (sample IDs) is included.
df_with_labels.to_csv(optimized_output_path, index_label="id_campione")

print(f"\n✅ Saved memory-optimized file: {optimized_output_path}")


In [ ]:
output_path_labels = "/kaggle/working/GSE69914_beta_with_labels_numeric.csv" ## CHANGE HERE!! ##

# Load the previously generated CSV (the first column is the index: CpG/ID_REF)
# I load the processed file where the first column (ID_REF) should be set as the DataFrame index.
df_with_labels = pd.read_csv(output_path_labels, index_col=0)

In [ ]:
df_with_labels.head()

In [ ]:
# Salvaggio dataset con float32 al posto di float64
# (facoltativo) riduci memoria
# df_T = df_T.astype("float32")